## Tarea 7 Métodos Numéricos
# Desafío Clima con Cadenas de Markov

En este desafío, usaremos cadenas de Markov para predecir patrones de clima en Chile, basado en datos históricos a lo largo de los años de parte de la Dirección Meteorológica de Chile.

Para esto, primero extraeremos los datos, y usaremos un DataFrame de Pandas para guardar esta información y usarla para crear cadenas de Markov.
Usaremos datos de la estación ubicada en Quinta Normal, con un código de 330020.
Originalmente, pensé usar datos de 1980 a 2025... sin embargo, creo que algo interesante sería lo siguiente: comparar los resultados obtenidos desde 1960 hasta 1991, a los resultados obtenidos desde 2015 a 2025. 

En los últimos 10 años, la temperatura promedio de cada día en Chile ha subido rápidamente. El propósito de este trabajo es **ver como cambian las cadenas de Markov si comparamos los datos antiguos con los datos de los últimos 10 años.** 

Para hacer esto, crearemos tres categorías de temperatura, analizando la temperatura maxima anual:<br>
Frio: **0-14°C**<br>
Templado: **15-28°C**<br>
Caluroso: **29°C**<br>


## Extraccion y preparacion de datos


In [51]:
import urllib.request as ureq
import pandas as pd
import numpy as np

variables = ["temperaturaMaximaAnual"] # lista en caso de agregar mas variables
years1 = list(range(1960, 1991))
years2 = list(range(2015, 2026))
estacion = 330020


urlbase = "https://climatologia.meteochile.gob.cl/application/anual"

# datos 1960 a 1990
datosYears1 = []
for v in variables:
    for a in years1:
        url = f"{urlbase}/{v}/{estacion}/{a}"
        try:
            with ureq.urlopen(url) as pagina:
                # Decodificamos a utf-8 para que sea string
                contenido = pagina.read().decode('utf-8')
                datosYears1.append({"variable": v, "year": a, "html": contenido})
        except Exception as e:
            print(f"Error en {a}: {e}")

# datos 2015 a 2025
datosYears2 = []
for v in variables:
    for a in years2:
        url = f"{urlbase}/{v}/{estacion}/{a}"
        try:
            with ureq.urlopen(url) as pagina:
                contenido = pagina.read().decode('utf-8')
                datosYears2.append({"variable": v, "year": a, "html": contenido})
        except Exception as e:
            print(f"Error en {a}: {e}")

In [56]:
def procesar_html_extraido(lista_html):
    registros_finales = []
    
    for entrada in lista_html:
        year = entrada['year']
        variable = entrada['variable']
        html = entrada['html']
        fragmentos = html.split('<td class="text-center">')

        fragmentos = fragmentos[1:]
        
        valores = [f.split('</td>')[0].strip() for f in fragmentos]
        
        for i in range(0, len(valores), 13):
            fila = valores[i : i + 13]
            if not fila: continue
            
            dia = fila[0] # primer valor corresponde al numero del dia
            
            # Recorremos los meses (posiciones 1 a 12)
            for mes_idx in range(1, 13):
                valor_clima = fila[mes_idx]
                
                # Filtrar celdas vacías, guiones o saltos de línea
                if valor_clima and valor_clima not in ['-', '', '&nbsp;', '\n']:
                    try:
                        registros_finales.append({
                            'fecha': f"{year}-{mes_idx:02d}-{int(dia):02d}",
                            'mes': mes_idx,
                            'temperatura': float(valor_clima),
                        })
                    except ValueError:
                        continue 
                        
    return registros_finales


datos_limpios_1 = procesar_html_extraido(datosYears1)
datos_limpios_2 = procesar_html_extraido(datosYears2) 

df1 = pd.DataFrame(datos_limpios_1)
df2 = pd.DataFrame(datos_limpios_2)

In [ ]:
'''
Frio: **0-14°C**
Templado: **15-28°C**
Caluroso: **29°C**
'''

df1['Estado'] = pd.cut(df1['temperatura'], bins=[-np.inf, 14, 28, np.inf], labels=['Frio', 'Templado', 'Caluroso'])
display(df1)

df2['Estado'] = pd.cut(df2['temperatura'], bins=[-np.inf, 14, 28, np.inf], labels=['Frio', 'Templado', 'Caluroso'])
display(df2)

,fecha,mes,temperatura,Estado
0,1960-01-01,1,21.8,Templado
1,1960-02-01,2,31.4,Caluroso
2,1960-03-01,3,30.1,Caluroso
3,1960-04-01,4,28.8,Caluroso
4,1960-05-01,5,24.4,Templado
...,...,...,...,...
11285,1990-05-31,5,16.4,Templado
11286,1990-07-31,7,21.6,Templado
11287,1990-08-31,8,10.8,Frio
11288,1990-10-31,10,24.2,Templado


,fecha,mes,temperatura,Estado
0,2015-01-01,1,30.2,Caluroso
1,2015-02-01,2,32.6,Caluroso
2,2015-03-01,3,29.8,Caluroso
3,2015-04-01,4,30.6,Caluroso
4,2015-05-01,5,19.4,Templado
...,...,...,...,...
4012,2025-05-31,5,20.9,Templado
4013,2025-07-31,7,15.3,Templado
4014,2025-08-31,8,15.2,Templado
4015,2025-10-31,10,23.8,Templado
